In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, List
from pathlib import Path
import glob
import os

class CryptoDataAnalyzer:
    """
    Utility class for analyzing collected cryptocurrency data
    """
    
    def __init__(self, snapshot_file: str = None, historical_file: str = None):
        """
        Initialize with data files
        
        Args:
            snapshot_file: Path to snapshot CSV (if None, auto-detects latest)
            historical_file: Path to historical CSV (if None, auto-detects latest)
        """
        # Auto-detect files if not provided
        if snapshot_file is None:
            snapshot_file = self._find_latest_file('crypto_snapshot_*.csv')
            if snapshot_file:
                print(f"📁 Auto-detected snapshot: {os.path.basename(snapshot_file)}")
        
        if historical_file is None:
            historical_file = self._find_latest_file('crypto_historical_ohlcv_*.csv')
            if historical_file:
                print(f"📁 Auto-detected historical: {os.path.basename(historical_file)}")
        
        # Load data
        if snapshot_file and os.path.exists(snapshot_file):
            self.snapshot_df = pd.read_csv(snapshot_file)
            print(f"✓ Loaded snapshot: {len(self.snapshot_df)} cryptocurrencies")
        else:
            print(f"⚠️  Snapshot file not found: {snapshot_file}")
            self.snapshot_df = pd.DataFrame()
        
        if historical_file and os.path.exists(historical_file):
            self.historical_df = pd.read_csv(historical_file)
            print(f"✓ Loaded historical: {len(self.historical_df)} records")
        else:
            self.historical_df = None
    
    def _find_latest_file(self, pattern: str, directory: str = "/mnt/user-data/outputs") -> str:
        """Find the most recent file matching pattern"""
        search_pattern = os.path.join(directory, pattern)
        files = glob.glob(search_pattern)
        
        if not files:
            return None
        
        # Return most recent file
        latest = max(files, key=os.path.getctime)
        return latest
    
    def quality_filter(self, 
                       min_data_completeness: float = 70,
                       max_zero_volume_pct: float = 20,
                       min_market_cap: float = 1e6,
                       min_avg_volume: float = 1e5) -> pd.DataFrame:
        """
        Filter for high-quality, liquid assets
        
        Returns:
            Filtered DataFrame with quality assets
        """
        df = self.snapshot_df.copy()
        
        filters = []
        
        # Data quality filters
        if 'data_completeness_pct' in df.columns:
            filters.append(df['data_completeness_pct'] >= min_data_completeness)
            
        if 'zero_volume_pct' in df.columns:
            filters.append(df['zero_volume_pct'] <= max_zero_volume_pct)
        
        # Market filters
        if 'marketCap' in df.columns:
            filters.append(df['marketCap'] >= min_market_cap)
        
        if 'avg_volume_30d' in df.columns:
            filters.append(df['avg_volume_30d'] >= min_avg_volume)
        
        # Apply all filters
        mask = pd.Series(True, index=df.index)
        for f in filters:
            mask = mask & f
        
        filtered = df[mask]
        
        print(f"\n🔍 Quality Filter Results:")
        print(f"   Original: {len(df)} cryptos")
        print(f"   Filtered: {len(filtered)} cryptos ({len(filtered)/len(df)*100:.1f}%)")
        print(f"   Removed: {len(df) - len(filtered)} cryptos")
        
        return filtered
    
    def calculate_correlation_matrix(self, 
                                    features: List[str] = None,
                                    min_valid: int = 50) -> pd.DataFrame:
        """
        Calculate correlation matrix for specified features
        
        Args:
            features: List of column names to correlate
            min_valid: Minimum number of valid pairs
        """
        if features is None:
            # Default features for correlation
            features = [
                'volatility_90d', 'beta_vs_btc', 'sharpe_ratio',
                'max_drawdown_pct', 'volume_mcap_ratio_pct'
            ]
        
        # Filter to available features
        available = [f for f in features if f in self.snapshot_df.columns]
        
        if not available:
            print("⚠️  No valid features found for correlation")
            return pd.DataFrame()
        
        print(f"\n📊 Calculating correlation matrix for {len(available)} features...")
        
        # Calculate correlation
        corr_df = self.snapshot_df[available].corr(min_periods=min_valid)
        
        return corr_df
    
    def find_similar_assets(self, 
                           symbol: str,
                           features: List[str] = None,
                           top_n: int = 10) -> pd.DataFrame:
        """
        Find assets most similar to a given symbol based on features
        
        Args:
            symbol: Target cryptocurrency symbol
            features: Features to use for similarity (None = auto)
            top_n: Number of similar assets to return
        """
        if symbol not in self.snapshot_df['symbol'].values:
            print(f"⚠️  Symbol '{symbol}' not found in dataset")
            return pd.DataFrame()
        
        if features is None:
            # Default similarity features
            features = [
                'volatility_90d', 'beta_vs_btc', 'sharpe_ratio',
                'max_drawdown_pct', 'vs_50ma_pct', 'vs_200ma_pct'
            ]
        
        # Filter to available features
        available = [f for f in features if f in self.snapshot_df.columns]
        
        if not available:
            print("⚠️  No valid features for similarity calculation")
            return pd.DataFrame()
        
        print(f"\n🔍 Finding assets similar to {symbol}...")
        print(f"   Using features: {', '.join(available)}")
        
        # Get target asset features
        df = self.snapshot_df.copy()
        target_features = df[df['symbol'] == symbol][available].iloc[0]
        
        # Calculate euclidean distance for all assets
        from sklearn.preprocessing import StandardScaler
        
        # Standardize features
        scaler = StandardScaler()
        scaled_features = scaler.fit_transform(df[available].fillna(0))
        
        # Target vector
        target_idx = df[df['symbol'] == symbol].index[0]
        target_vector = scaled_features[target_idx]
        
        # Calculate distances
        distances = np.sqrt(((scaled_features - target_vector) ** 2).sum(axis=1))
        df['similarity_distance'] = distances
        
        # Get most similar (excluding self)
        similar = df[df['symbol'] != symbol].nsmallest(top_n, 'similarity_distance')
        
        result = similar[['symbol', 'name', 'marketCap', 'similarity_distance'] + available]
        
        return result
    
    def cluster_assets(self, 
                      n_clusters: int = 6,
                      features: List[str] = None) -> pd.DataFrame:
        """
        Cluster cryptocurrencies into groups
        
        Args:
            n_clusters: Number of clusters
            features: Features for clustering
        """
        if features is None:
            features = [
                'volatility_90d', 'beta_vs_btc', 'sharpe_ratio',
                'max_drawdown_pct', 'volume_mcap_ratio_pct'
            ]
        
        available = [f for f in features if f in self.snapshot_df.columns]
        
        if not available:
            print("⚠️  No valid features for clustering")
            return pd.DataFrame()
        
        print(f"\n🎯 Clustering {len(self.snapshot_df)} assets into {n_clusters} groups...")
        print(f"   Features: {', '.join(available)}")
        
        from sklearn.preprocessing import StandardScaler
        from sklearn.cluster import KMeans
        
        # Prepare data
        df = self.snapshot_df.copy()
        X = df[available].fillna(df[available].median())
        
        # Standardize
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        # Cluster
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        df['cluster'] = kmeans.fit_predict(X_scaled)
        
        # Cluster profiles
        print("\n📊 Cluster Profiles:")
        cluster_summary = df.groupby('cluster').agg({
            'symbol': 'count',
            **{f: 'mean' for f in available}
        })
        cluster_summary.columns = ['count'] + [f'avg_{f}' for f in available]
        
        print(cluster_summary.to_string())
        
        return df
    
    def sector_analysis(self, sector_column: str = 'mcap_category') -> pd.DataFrame:
        """
        Analyze metrics by sector/category
        
        Args:
            sector_column: Column to group by
        """
        if sector_column not in self.snapshot_df.columns:
            print(f"⚠️  Column '{sector_column}' not found")
            return pd.DataFrame()
        
        print(f"\n📈 Sector Analysis by {sector_column}:")
        
        metrics = [
            'price', 'marketCap', 'volume', 'changePercentage',
            'volatility_90d', 'sharpe_ratio', 'beta_vs_btc'
        ]
        
        available = [m for m in metrics if m in self.snapshot_df.columns]
        
        sector_stats = self.snapshot_df.groupby(sector_column)[available].agg(['mean', 'median', 'count'])
        
        return sector_stats
    
    def liquidity_summary(self) -> Dict:
        """Generate liquidity summary statistics"""
        print("\n💧 Liquidity Summary:")
        
        summary = {}
        
        if 'avg_volume_30d' in self.snapshot_df.columns:
            summary['total_30d_volume'] = self.snapshot_df['avg_volume_30d'].sum()
            summary['median_30d_volume'] = self.snapshot_df['avg_volume_30d'].median()
            summary['high_liquidity_count'] = (self.snapshot_df['avg_volume_30d'] > 1e6).sum()
        
        if 'zero_volume_days' in self.snapshot_df.columns:
            summary['avg_zero_volume_days'] = self.snapshot_df['zero_volume_days'].mean()
            summary['pct_with_zero_days'] = (self.snapshot_df['zero_volume_days'] > 0).sum() / len(self.snapshot_df) * 100
        
        if 'data_completeness_pct' in self.snapshot_df.columns:
            summary['avg_data_completeness'] = self.snapshot_df['data_completeness_pct'].mean()
            summary['high_quality_data_pct'] = (self.snapshot_df['data_completeness_pct'] > 80).sum() / len(self.snapshot_df) * 100
        
        for key, value in summary.items():
            print(f"   {key}: {value:,.2f}")
        
        return summary
    
    def risk_profile(self, top_n: int = 20) -> pd.DataFrame:
        """
        Create risk profile for assets
        
        Returns:
            DataFrame with risk metrics for top assets by market cap
        """
        print(f"\n⚠️  Risk Profile (Top {top_n} by Market Cap):")
        
        risk_metrics = [
            'symbol', 'name', 'marketCap', 'volatility_90d',
            'max_drawdown_pct', 'beta_vs_btc', 'sharpe_ratio'
        ]
        
        available = [m for m in risk_metrics if m in self.snapshot_df.columns]
        
        # Get top assets
        top_assets = self.snapshot_df.nlargest(top_n, 'marketCap')[available]
        
        # Add risk score (simple weighted combination)
        if all(m in top_assets.columns for m in ['volatility_90d', 'max_drawdown_pct']):
            # Higher is riskier
            vol_normalized = (top_assets['volatility_90d'] - top_assets['volatility_90d'].min()) / \
                           (top_assets['volatility_90d'].max() - top_assets['volatility_90d'].min())
            dd_normalized = abs((top_assets['max_drawdown_pct'] - top_assets['max_drawdown_pct'].min()) / \
                            (top_assets['max_drawdown_pct'].max() - top_assets['max_drawdown_pct'].min()))
            
            top_assets['risk_score'] = ((vol_normalized + dd_normalized) / 2 * 100).round(2)
        
        return top_assets
    
    def export_analysis(self, output_file: str = 'crypto_analysis_results.csv'):
        """Export analyzed/filtered data"""
        # Combine all useful metrics
        export_df = self.snapshot_df.copy()
        
        filepath = f"../outputs/{output_file}"
        export_df.to_csv(filepath, index=False)
        
        print(f"\n💾 Analysis exported to: {output_file}")
        return filepath


def example_analysis():
    """
    Example analysis workflow
    """
    print("=" * 80)
    print("CRYPTOCURRENCY DATA ANALYSIS EXAMPLE")
    print("=" * 80)
    
    # Auto-detect latest files (or specify manually)
    # analyzer = CryptoDataAnalyzer()  # Auto-detect both files
    # OR specify manually:
    # analyzer = CryptoDataAnalyzer(
    #     snapshot_file="/mnt/user-data/outputs/crypto_snapshot_20260208_123456.csv",
    #     historical_file="/mnt/user-data/outputs/crypto_historical_ohlcv_20260208_123456.csv"
    # )
    
    analyzer = CryptoDataAnalyzer()  # Auto-detect
    
    if analyzer.snapshot_df.empty:
        print("\n⚠️  No data found. Please run advanced_crypto_collector.py first!")
        return
    
    # 1. Quality filtering
    quality_assets = analyzer.quality_filter(
        min_data_completeness=70,
        max_zero_volume_pct=20,
        min_market_cap=10e6,
        min_avg_volume=1e5
    )
    
    # 2. Liquidity summary
    liquidity_stats = analyzer.liquidity_summary()
    
    # 3. Find similar assets to Bitcoin
    similar_to_btc = analyzer.find_similar_assets('BTCUSD', top_n=10)
    print("\n🔍 Assets similar to Bitcoin:")
    print(similar_to_btc[['symbol', 'name', 'marketCap', 'similarity_distance']].to_string(index=False))
    
    # 4. Cluster analysis
    clustered = analyzer.cluster_assets(n_clusters=6)
    
    # 5. Risk profiling
    risk_profile = analyzer.risk_profile(top_n=20)
    print(risk_profile.to_string(index=False))
    
    # 6. Sector analysis
    sector_stats = analyzer.sector_analysis('mcap_category')
    print("\n📊 Sector Statistics:")
    print(sector_stats)
    
    # 7. Correlation matrix
    corr_matrix = analyzer.calculate_correlation_matrix()
    print("\n🔗 Feature Correlations:")
    print(corr_matrix.to_string())
    
    print("\n" + "=" * 80)
    print("ANALYSIS COMPLETE")
    print("=" * 80)


if __name__ == "__main__":
    example_analysis()